In [2]:
import csv

def ler_transacoes():

    nome_arquivo = 'transacoes.csv'
    transacoes_brutas = []

    try:
        # 'with open' abre o arquivo e garante
        # que ele será fechado automaticamente quando terminar, mesmo se der erro.
        with open(nome_arquivo, mode='r', encoding='utf-8') as arquivo:
            # csv.DictReader lê cada linha do CSV e transforma em um dicionário

            leitor = csv.DictReader(arquivo)

            for linha in leitor:
                transacoes_brutas.append(linha)

        return transacoes_brutas

    except FileNotFoundError:
        print(f"Erro: O arquivo '{nome_arquivo}' não foi encontrado.")
        return []

# --- TESTE DA FUNÇÃO ---
dados_brutos = ler_transacoes()
print(f"Total de linhas brutas lidas do arquivo: {len(dados_brutos)}")

Total de linhas brutas lidas do arquivo: 20


In [4]:
from datetime import datetime

def validar_transacao(linha):
    """
    Se for válida, retorna o dicionário com os tipos corrigidos (int, float, datetime).
    Se for inválida, retorna None.
    """
    try:
        # 1. Validação do ID: Não pode ser vazio e precisa ser numérico
        id_texto = linha.get('id', '').strip()
        if not id_texto or not id_texto.isdigit():
            return None
        id_valido = int(id_texto)

        # 2. Validação do cliente_id: Não pode ser vazio
        cliente_id = linha.get('cliente_id', '').strip()
        if not cliente_id:
            return None

        # 3. Validação do tipo: Precisa ser obrigatoriamente 'credito' ou 'debito'
        tipo = linha.get('tipo', '').strip().lower()
        if tipo not in ['credito', 'debito']:
            return None

        # 4. Validação do valor: Tenta converter para float (decimal) e precisa ser > 0
        valor_texto = linha.get('valor', '').strip()
        # try/except interno para capturar se tentarem passar letras (ex: 'abc')
        try:
            valor_valido = float(valor_texto)
            if valor_valido <= 0:
                return None
        except ValueError:
            return None # Cai aqui se a conversão para float falhar (ex: float('abc'))

        # 5. Validação da data: Precisa estar no formato AAAA-MM-DD
        data_texto = linha.get('data', '').strip()
        try:
            # datetime.strptime tenta encaixar o texto no padrão informado.
            # Se for diferente (ou vazio), ele joga um ValueError.
            data_valida = datetime.strptime(data_texto, "%Y-%m-%d")
        except ValueError:
            return None

        # Se passou por TODOS os filtros acima "objeto" limpo e tipado
        return {
            "id": id_valido,
            "data": data_valida,
            "cliente_id": cliente_id,
            "tipo": tipo,
            "valor": valor_valido,
            "descricao": linha.get('descricao', '').strip(),
            "categoria": linha.get('categoria', '').strip()
        }

    except Exception:
        # Qualquer outro erro, descartamos a linha
        return None

In [5]:
# Lista guardar apenas as transações que passarem no teste
transacoes_limpas = []
total_invalidas = 0

for linha_bruta in dados_brutos:
    linha_validada = validar_transacao(linha_bruta)

    if linha_validada is not None:
        transacoes_limpas.append(linha_validada)
    else:
        total_invalidas += 1

print(f"Total de linhas lidas: {len(dados_brutos)}")
print(f"Linhas válidas: {len(transacoes_limpas)}")
print(f"Linhas inválidas: {total_invalidas}")

Total de linhas lidas: 20
Linhas válidas: 15
Linhas inválidas: 5


In [6]:
def gerar_relatorio(transacoes_limpas):
    """
    Processar a lista de transações válidas para gerar as métricas mensais,
    calcular o período do relatório e listar transações suspeitas.
    """
    # 1. Definindo a constante de suspeita exigida pelo enunciado
    LIMITE_SUSPEITO = 10000.00

    # Estruturas para guardar nossos resultados
    resumo_mensal = {}
    transacoes_suspeitas = []

    # Listas auxiliares só para descobrir a data mais antiga e a mais recente
    todas_datas = []

    # 2. Varrendo cada transação limpa
    for t in transacoes_limpas:
        todas_datas.append(t['data'])

        # Extraindo o mês no formato AAAA-MM a partir do objeto datetime
        ano_mes = t['data'].strftime("%Y-%m-%m") # Ex: '2026-01'

        # Se for a primeira vez que vemos esse mês, inicializamos o dicionário dele
        if ano_mes not in resumo_mensal:
            resumo_mensal[ano_mes] = {
                "quantidade": 0,
                "total_credito": 0.0,
                "total_debito": 0.0,
                "maior_valor": 0.0,
                "menor_valor": float('inf'),
                "valores": [] # Lista auxiliar para calcular a média depois
            }

        # Atalho para o mês atual no loop
        mes_atual = resumo_mensal[ano_mes]

        # Contabilizando a transação
        mes_atual["quantidade"] += 1
        valor = t['valor']
        mes_atual["valores"].append(valor)

        # Acumulando por tipo (credito ou debito)
        if t['tipo'] == 'credito':
            mes_atual["total_credito"] += valor
        elif t['tipo'] == 'debito':
            mes_atual["total_debito"] += valor

        # Verificando maior e menor valor do mês
        if valor > mes_atual["maior_valor"]:
            mes_atual["maior_valor"] = valor
        if valor < mes_atual["menor_valor"]:
            mes_atual["menor_valor"] = valor

        # 3. Verificação de Transação Suspeita (Requisito de Fraude)
        if valor > LIMITE_SUSPEITO:
            transacoes_suspeitas.append({
                "id": t['id'],
                "cliente_id": t['cliente_id'],
                "data": t['data'].strftime("%Y-%m-%d"),
                "valor": valor
            })

    # 4. Cálculos (Saldo, Média e Período Geral)
    for ano_mes, dados in resumo_mensal.items():
        dados["saldo"] = dados["total_credito"] - dados["total_debito"]
        dados["media"] = sum(dados["valores"]) / dados["quantidade"]
        # Remover a lista auxiliar para não poluir o resultado final
        del dados["valores"]

    # Calcular a distância em dias entre a transação mais antiga e a mais recente
    if todas_datas:
        data_mais_antiga = min(todas_datas)
        data_mais_recente = max(todas_datas)
        dias_totais = (data_mais_recente - data_mais_antiga).days
    else:
        data_mais_antiga, data_mais_recente, dias_totais = None, None, 0

    # Retornar todas as métricas estruturadas
    return {
        "periodo": {
            "inicio": data_mais_antiga.strftime("%Y-%m-%d") if data_mais_antiga else "N/A",
            "fim": data_mais_recente.strftime("%Y-%m-%d") if data_mais_recente else "N/A",
            "dias_passados": dias_totais
        },
        "resumo_mensal": resumo_mensal,
        "transacoes_suspeitas": transacoes_suspeitas
    }

In [7]:
dados_processados = gerar_relatorio(transacoes_limpas)

print(f"Período analisado: {dados_processados['periodo']['inicio']} até {dados_processados['periodo']['fim']}")
print(f"Dias decorridos no período: {dados_processados['periodo']['dias_passados']} dias")
print(f"Quantidade de transações suspeitas: {len(dados_processados['transacoes_suspeitas'])}")

Período analisado: 2026-01-05 até 2026-03-25
Dias decorridos no período: 79 dias
Quantidade de transações suspeitas: 2


In [10]:
import json

def exibir_relatorio(dados_processados, total_lidas, total_validas, total_invalidas):
    """
    Formata e imprime o relatório financeiro e os alertas de fraude no terminal.
    """
    print("\n" + "="*30)
    print("       RESUMO DA LIMPEZA       ")
    print("="*30)
    print(f"Total de linhas lidas: {total_lidas}")
    print(f"Linhas válidas:       {total_validas}")
    print(f"Linhas inválidas:     {total_invalidas}")
    print(f"Período analisado:    {dados_processados['periodo']['inicio']} → {dados_processados['periodo']['fim']} ({dados_processados['periodo']['dias_passados']} dias)")

    print("\n" + "="*30)
    print("       RELATÓRIO MENSAL       ")
    print("="*30)

    # Varre cada mês calculado
    for mes, metricas in dados_processados["resumo_mensal"].items():
        print(f"\nMês: {mes}")
        print(f"  Transações:  {metricas['quantidade']}")
        print(f"  Total crédito: R$ {metricas['total_credito']:,.2f}".replace(",", "X").replace(".", ",").replace("X", "."))
        print(f"  Total débito:  R$ {metricas['total_debito']:,.2f}".replace(",", "X").replace(".", ",").replace("X", "."))
        print(f"  Saldo:         R$ {metricas['saldo']:,.2f}".replace(",", "X").replace(".", ",").replace("X", "."))
        print(f"  Média:         R$ {metricas['media']:,.2f}".replace(",", "X").replace(".", ",").replace("X", "."))
        print(f"  Maior valor:   R$ {metricas['maior_valor']:,.2f}".replace(",", "X").replace(".", ",").replace("X", "."))
        print(f"  Menor valor:   R$ {metricas['menor_valor']:,.2f}".replace(",", "X").replace(".", ",").replace("X", "."))

    print("\n" + "="*30)
    print("     TRANSAÇÕES SUSPEITAS     ")
    print("="*30)

    suspeitas = dados_processados["transacoes_suspeitas"]
    if not suspeitas:
        print("Nenhuma transação suspeita encontrada.")
    else:
        for s in suspeitas:
            valor_fmt = f"R$ {s['valor']:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")
            print(f"ID: {s['id']} | Cliente: {s['cliente_id']} | Data: {s['data']} | Valor: {valor_fmt}")
    print("="*30 + "\n")


def salvar_json(dados_processados, total_validas, total_invalidas):
    """
    Gera o arquivo 'relatorio.json' com a estrutura exigida pelo desafio.
    """
    nome_arquivo = "relatorio.json"
    data_hoje_string = datetime.now().strftime("%Y-%m-%d")

    estrutura_json = {
        "gerado_em": data_hoje_string,
        "total_transacoes_validas": total_validas,
        "total_transacoes_invalidas": total_invalidas,
        "resumo_mensal": {}
    }

    # Copia os dados do resumo mensal
    for mes, metricas in dados_processados["resumo_mensal"].items():
        estrutura_json["resumo_mensal"][mes] = {
            "quantidade": metricas["quantidade"],
            "total_credito": round(metricas["total_credito"], 2),
            "total_debito": round(metricas["total_debito"], 2),
            "saldo": round(metricas["saldo"], 2),
            "media": round(metricas["media"], 2),
            "maior_valor": round(metricas["maior_valor"], 2),
            "menor_valor": round(metricas["menor_valor"], 2)
        }

    try:
        with open(nome_arquivo, mode='w', encoding='utf-8') as arquivo:
            json.dump(estrutura_json, arquivo, ensure_ascii=False, indent=2)
        print(f"Arquivo '{nome_arquivo}' gerado com sucesso!")
    except Exception as e:
        print(f"Erro ao salvar o arquivo JSON: {e}")

In [11]:
def executar_pipeline_financeiro():
    # 1. Ler os dados
    linhas_brutas = ler_transacoes()
    if not linhas_brutas:
        print("Execução interrompida: Nenhum dado encontrado para processar.")
        return

    # 2. Validar e Limpar
    transacoes_limpas = []
    total_invalidas = 0
    for linha in linhas_brutas:
        linha_validada = validar_transacao(linha)
        if linha_validada is not None:
            transacoes_limpas.append(linha_validada)
        else:
            total_invalidas += 1

    total_lidas = len(linhas_brutas)
    total_validas = len(transacoes_limpas)

    # 3. Processar Métricas e Datas
    dados_processados = gerar_relatorio(transacoes_limpas)

    # 4. Exibir Relatório no Terminal
    exibir_relatorio(dados_processados, total_lidas, total_validas, total_invalidas)

    # 5. Salvar o arquivo JSON
    salvar_json(dados_processados, total_validas, total_invalidas)

# Ativa o programa completo
executar_pipeline_financeiro()


       RESUMO DA LIMPEZA       
Total de linhas lidas: 20
Linhas válidas:       15
Linhas inválidas:     5
Período analisado:    2026-01-05 → 2026-03-25 (79 dias)

       RELATÓRIO MENSAL       

Mês: 2026-01-01
  Transações:  4
  Total crédito: R$ 15.500,00
  Total débito:  R$ 225,50
  Saldo:         R$ 15.274,50
  Média:         R$ 3.931,38
  Maior valor:   R$ 12.000,00
  Menor valor:   R$ 45,00

Mês: 2026-02-02
  Transações:  3
  Total crédito: R$ 15.000,00
  Total débito:  R$ 409,90
  Saldo:         R$ 14.590,10
  Média:         R$ 5.136,63
  Maior valor:   R$ 15.000,00
  Menor valor:   R$ 89,90

Mês: 2026-03-03
  Transações:  8
  Total crédito: R$ 4.060,00
  Total débito:  R$ 2.729,90
  Saldo:         R$ 1.330,10
  Média:         R$ 848,74
  Maior valor:   R$ 3.500,00
  Menor valor:   R$ 60,00

     TRANSAÇÕES SUSPEITAS     
ID: 5 | Cliente: CLI003 | Data: 2026-02-14 | Valor: R$ 15.000,00
ID: 11 | Cliente: CLI005 | Data: 2026-01-22 | Valor: R$ 12.000,00

Arquivo 'relatorio.json' 